In [6]:
import pandas as pd
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [2]:
#상관계수 데이터 프레임 불러기기
cor=pd.read_csv('data/모든_상관계수_결측치_제거.csv')
idx=cor[cor['Correlation'].abs()>0.4].index
cor=cor.loc[idx]
check_col=set(cor['Variable'].values)

In [3]:
train_path='data/train/parquet'

df_list=[]
for x in os.listdir(train_path):
    file_path=os.path.join(train_path,x)
    
    df=pd.read_parquet(file_path)
    df=df[list(set(df.columns) & check_col|{'ID','기준년월'})]
    print(df.columns)
    
    
    gc.collect()
    
    df_list.append(df)
df=pd.merge(left=df_list[0],right=df_list[1],on=['ID','기준년월'])
for x in range(7):
    df=pd.merge(left=df,right=df_list[x+1],on=['ID','기준년월'])
    gc.collect()
#'ID','기준년월' 칼럼 제거
df=df.drop(['ID','기준년월'],axis=1)
df.to_csv('data/fiture.csv',encoding='utf-8-sig',index=False)

Index(['_2순위카드이용금액', '이용금액_R3M_신용체크', '기준년월', 'ID', '이용금액_R3M_신용',
       '_1순위카드이용금액'],
      dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['이용건수_신판_R6M', '이용금액_일시불_B0M', '정상청구원금_B0M', '이용건수_신용_R6M',
       '이용금액_일시불_R3M', '이용건수_신판_R12M', '기준년월', '이용건수_신용_B0M', '쇼핑_도소매_이용금액',
       '이용건수_일시불_R12M', '정상입금원금_B2M', '이용건수_오프라인_B0M', '_1순위교통업종_이용금액',
       '이용건수_일시불_R3M', '이용건수_신용_R3M', '_2순위쇼핑업종_이용금액', '연체입금원금_B0M',
       '이용건수_일시불_B0M', '정상입금원금_B5M', '이용금액_일시불_R12M', '_3순위업종_이용금액',
       '이용건수_신판_R3M', '이용건수_일시불_R6M', '이용금액_일시불_R6M', '이용건수_오프라인_R3M',
       '이용건수_신용_R12M', '정상청구원금_B2M', '이용금액_오프라인_R6M', '이용가맹점수',
       '_3순위쇼핑업종_이용금액', '정상청구원금_B5M', '이용건수_오프라인_R6M', '이용금액_오프라인_B0M',
       '이용건수_신판_B0M', '최대이용금액_일시불_R12M', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '_2순위업종_이용금액', '_1순위업종_이용금액', 'ID'],
      dtype='object')
Index(['청구금액_B0', '기준년월', '청구금액_R6M', 'ID', '청구금액_R3M'], dtype='object')
Index(['평잔_일시불_6M', '월중평잔_일시불', '월중평잔_일시불_B0M', '기준년월', 'ID', '평잔_일시불_3M',
    

In [8]:
df=pd.read_csv('data/fiture.csv')
#범주형 데이터 인코딩
for x in df.select_dtypes(include=["object"]).columns:
    #각 인코더 별로 변수에 저장    
    globals()[f'{x}_encoder'] = LabelEncoder()
    eval(f'{x}_encoder').fit(df[x])
    df[x] = eval(f'{x}_encoder').transform(df[x])
   

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

#최후에 남길 피쳐의 개수
last_fit_num= 5


# VIF 계산을 위한 데이터프레임 복사
vif_X = df.copy()

# 피처를 제거해 나갈 개수
num_features_to_remove = len(vif_X.columns) - last_fit_num
for i in range(num_features_to_remove):
    # 1. 상수항 추가
    X_with_const = sm.add_constant(vif_X)

    # 2. VIF 계산
    vif = pd.DataFrame()
    vif["변수"] = X_with_const.columns
    vif["VIF"] = [variance_inflation_factor(X_with_const.values, j) for j in range(X_with_const.shape[1])]
    
    # 3. 상수항(const)을 제외한 VIF 데이터프레임 생성
    vif_features = vif[vif['변수'] != 'const']
    
    # 4. VIF가 가장 높은 피처 찾기
    max_vif_feature = vif_features.loc[vif_features['VIF'].idxmax()]
    
    # 제거될 피처 정보 출력
    print(f"Iteration {i+1}:")
    print(f"제거될 피처: '{max_vif_feature['변수']}' (VIF: {max_vif_feature['VIF']:.4f})")
    
    # 5. 해당 피처 제거
    vif_X = vif_X.drop(max_vif_feature['변수'], axis=1)
    
    # CSV로 현재 상태 저장 (선택 사항)
    vif.to_csv(f'data/vif_step_{len(df.columns)-i-1}.csv', encoding='utf-8-sig', index=False)
    
    print(f"남은 피처 개수: {len(vif_X.columns)}개")
    print("-" * 50)
    
    gc.collect()

print("최종 선택된 피처:")
print(vif_X.columns.tolist())

Iteration 1:
제거될 피처: '이용건수_신판_R6M' (VIF: 107585.3366)
남은 피처 개수: 49개
--------------------------------------------------
Iteration 2:
제거될 피처: '이용건수_신판_R3M' (VIF: 63224.2482)
남은 피처 개수: 48개
--------------------------------------------------
Iteration 3:
제거될 피처: '이용건수_신판_R12M' (VIF: 34029.9779)
남은 피처 개수: 47개
--------------------------------------------------
Iteration 4:
제거될 피처: '이용건수_일시불_R3M' (VIF: 20672.4591)
남은 피처 개수: 46개
--------------------------------------------------
Iteration 5:
제거될 피처: '이용건수_신용_R6M' (VIF: 12333.7925)
남은 피처 개수: 45개
--------------------------------------------------
Iteration 6:
제거될 피처: '이용건수_신판_B0M' (VIF: 10271.5323)
남은 피처 개수: 44개
--------------------------------------------------
Iteration 7:
제거될 피처: '이용건수_일시불_R12M' (VIF: 4881.3469)
남은 피처 개수: 43개
--------------------------------------------------
Iteration 8:
제거될 피처: '이용건수_신용_B0M' (VIF: 2564.6241)
남은 피처 개수: 42개
--------------------------------------------------
Iteration 9:
제거될 피처: '이용건수_신용_R3M' (VIF: 134.2273)
남은

In [ ]:
df1_list=[]
for i,x in enumerate(sorted([x for x in os.listdir('data') if x[:9]=='vif_step_'])):
    
    df1= pd.read_csv(f"data/{x}",encoding='utf-8-sig',index_col='변수')       
    df1.columns=[f'{i+last_fit_num}개']
    df1_list.append(df1)
total=pd.concat(df1_list,axis=1)
total=total.drop('const')
total.to_csv('data/total_vif.csv',encoding='utf-8-sig')